In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
import pickle

In [2]:
data_dir = "../data"
x_list = []
y_list = []
batch_size = 32

In [3]:
for file in os.listdir(data_dir):
    path = os.path.join(data_dir, file)
    with open(path, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')

        x_list.append(data_dict[b'data'])
        y_list.extend(data_dict[b'labels'])

C:\Users\xavif\AppData\Local\Temp\ipykernel_12608\250140131.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data_dict = pickle.load(fo, encoding='bytes')


In [4]:
X = np.concatenate(x_list)
Y = np.array(y_list)

In [5]:
X = X.reshape(-1, 3,32,32)
X = X.transpose(0,2,3,1)
X = X.astype(float) / 255.0

In [6]:
y = to_categorical(Y, 10)

In [7]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = dataset.shuffle(buffer_size=len(X), seed=42)

In [8]:
train_size = int(0.8*len(X))

In [9]:
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

In [10]:
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [11]:
print(len(X), X.shape)
print(len(y), y.shape if isinstance(y, np.ndarray) else 'no es ndarray')

50000 (50000, 32, 32, 3)
50000 (50000, 10)


In [12]:
IMG_SIZE = (32,32) + (3,)

In [13]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')

C:\Users\xavif\AppData\Local\Temp\ipykernel_12608\846687965.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')


In [ ]:
base_model.summary()

# Congelación de capas

In [20]:
base_model.trainable = True

In [21]:
for layer in base_model.layers[:50]:
    layer.trainable = False

In [22]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x= layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

In [23]:
model = models.Model(inputs=base_model.input, outputs=output)

In [24]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [25]:
history = model.fit(
    train_dataset, 
    validation_data=val_dataset,
    epochs=10

)

Epoch 1/10


c:\Users\xavif\Documents\MASTER\MobileNet-TransferLearning\.venv\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1172: UserWarning: "`categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 129s 90ms/step - accuracy: 0.5605 - loss: 1.3122 - val_accuracy: 0.5956 - val_loss: 1.5126
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.6757 - loss: 0.9809 - val_accuracy: 0.6444 - val_loss: 1.3351
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 130s 104ms/step - accuracy: 0.7131 - loss: 0.8771 - val_accuracy: 0.7270 - val_loss: 0.9597
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 126s 101ms/step - accuracy: 0.7348 - loss: 0.8100 - val_accuracy: 0.6823 - val_loss: 1.2457
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 116s 92ms/step - accuracy: 0.7480 - loss: 0.7611 - val_accuracy: 0.7233 - val_loss: 0.8841
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 113s 91ms/step - accuracy: 0.7685 - loss: 0.6997 - val_accuracy: 0.7462 - val_loss: 0.8731
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 116s 93ms/step - accuracy: 0.7832 - loss: 0.6626 - val_accuracy: 0.7673 - val_loss: 0.7431
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 120s 96ms/step - accuracy: 0.7942 - 